In [ ]:
import subprocess, sys, os, shutil
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER','0')
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=False)
UV=shutil.which('uv') or 'uv'
def run(cmd,phase):
    p=subprocess.run(cmd,capture_output=True,text=True)
    print(phase,'-> exit',p.returncode)
    if p.returncode!=0:
        print(p.stdout[-1200:]); print(p.stderr[-1200:]); raise RuntimeError(phase+' failed')
R=['torch>=2.8.0','triton>=3.4.0','transformers==4.56.2','peft==0.20.0','trl==0.22.2',
   'datasets==5.0.1','accelerate==1.15.0','bitsandbytes==0.50.2','openai-harmony==0.0.8']
run([UV,'pip','install','--system','--python',sys.executable,'--no-cache-dir',*R],'resolver')
run([UV,'pip','install','--system','--python',sys.executable,'--no-cache-dir','--no-deps','--upgrade',
     'unsloth==2026.9.4','unsloth_zoo==2026.9.3'],'frozen-no-deps')
print('install complete')


In [ ]:
# GHARIBO EXP-002 — 3-PROMPT PERFORMANCE BENCHMARK (NO SCORING, NO GOLD)
#
# ROOT CAUSE OF THE >900s/ITEM INCIDENT (fixed here):
#   1. NO stopping criterion. `generate()` was called with max_new_tokens=3072 and
#      no Harmony terminator handling, so it ran to the FULL 3072-token ceiling
#      every time instead of stopping at <|return|>. That alone is ~20-40x the
#      ~600-1100 tokens a gold answer actually needs.
#   2. TWO-GPU SHARDING. Unsloth split weights 5.68 GiB per T4 with `lm_head` on
#      cuda:1, so every generated token crossed the PCIe bus for the LM head.
#      The model fits on ONE T4, so sharding only cost time.
#   3. torch.no_grad() instead of torch.inference_mode().
#
# These are RUNTIME fixes only: adapter, base model, prompts, template and
# generation semantics (greedy) are unchanged.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # FIX 2: single GPU, no cross-device lm_head
import json, time, torch, pathlib, re

PROMPTS = json.loads(r'''[{"item_id": "8babd32b23f68f892a11a994354bb0b9f31faf8311baed14f958d57f100ef48b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:b3a3d33af19ba0fb) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:b3a3d33af19ba0fb\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:ubiquiti:unifi-access-current-readers-and-intercoms:g3-intercom\",\n      \"targetExternalKey\": \"brand:unifi\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:ubi-access-readers\",\n        \"sourceUrl\": \"https://www.ui.com/us/en/door-access/readers\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BRANDED_BY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "53de9c24696f4dc6696448b797e82d6cf8f6250735a0e809bcec925dbcde1043", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:security:requirements-assessment) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:security:requirements-assessment\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Security Requirements Assessment\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SERVICE\",\n      \"serviceClass\": \"CONSULTING\",\n      \"purpose\": \"Establish functional, operational and security requirements before system design.\",\n      \"typicalActivities\": [\n        \"Review business/security goals\",\n        \"Identify required functions and constraints\",\n        \"Document design inputs\"\n      ],\n      \"typicalDeliverables\": [\n        \"Requirements basis / assessment notes\"\n      ],\n      \"applicableSystemGroups\": [\n        \"ALL_SECURITY_SYSTEMS\"\n      ],\n      \"deliveryModes\": [\n        \"On-site\",\n        \"Remote\"\n      ],\n      \"recurring\": false,\n      \"prerequisites\": [],\n      \"vendorNeutralRegistryService\": true,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"service_class\",\n        \"name\": \"Service class\",\n        \"value\": \"CONSULTING\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:services:assessment-design\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recurring\",\n        \"name\": \"Recurring service\",\n        \"value\": false,\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"boolean\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:9851c0cef71e510286dccef3\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/solutions/professional-services\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Security Requirements Assessment.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "24e0a29c81ce87e41e89a3e92f56f16807faea3dd04b85d456dd9698ec6fd22c", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:bispectral-thermal-visible-surveillance) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:bispectral-thermal-visible-surveillance\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Bispectral Thermal + Visible Surveillance System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"MULTISENSOR_ARCHITECTURE\",\n      \"purpose\": \"Integrated thermal-and-visible architecture combining heat-based detection with visual verification.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": \"Thermal + visible video\",\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Bispectral camera\",\n        \"VMS/NVR/cloud endpoint\",\n        \"Network\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Management support for selected bispectral device/functions\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"MULTISENSOR_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:specialized-video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"signal_transport\",\n        \"name\": \"Signal / transport\",\n        \"value\": \"Thermal + visible video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:c2c37b4cd66c1ca67f4d11f3\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/products/thermal-cameras\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Bispectral Thermal + Visible Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}]''')
assert all(set(p.keys()) == {'item_id','messages'} for p in PROMPTS), 'gold leaked into payload'
print('benchmark prompts:', len(PROMPTS), '(subset of the frozen 20; gold NEVER present)')

WORKING = pathlib.Path('/kaggle/working')
ADAPTER = None
for c in sorted(pathlib.Path('/kaggle/input').rglob('adapter_model.safetensors')):
    if 'checkpoint' not in str(c):
        ADAPTER = c.parent; break
assert ADAPTER is not None, 'adapter not found'
print('adapter dir:', ADAPTER)

t0 = time.time()
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER), max_seq_length=3072, load_in_4bit=True, full_finetuning=False)
ADAPTER_LOAD = time.time() - t0
MODEL_LOAD = ADAPTER_LOAD

FastLanguageModel.for_inference(model)
model.eval()
print('device map:', getattr(model, 'hf_device_map', 'n/a'))
print('lm_head device:', next(model.lm_head.parameters()).device if hasattr(model,'lm_head') else 'n/a')

# FIX 1: explicit Harmony terminators + stopping criteria.
TERMINATORS = ['<|return|>', '<|call|>']
STOP_IDS = set()
for tok in TERMINATORS:
    ids = tokenizer.encode(tok, add_special_tokens=False)
    if ids: STOP_IDS.add(ids[-1])
if tokenizer.eos_token_id is not None: STOP_IDS.add(tokenizer.eos_token_id)
print('stop token ids:', sorted(STOP_IDS))
from transformers import StoppingCriteria, StoppingCriteriaList
class HarmonyStop(StoppingCriteria):
    def __init__(self, ids): self.ids = ids
    def __call__(self, input_ids, scores, **kw):
        return all(int(input_ids[i][-1]) in self.ids for i in range(input_ids.shape[0]))

def render(msgs):
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
        reasoning_effort='medium', strftime_now=lambda _f: '2026-09-15')

def final_channel(text):
    last = None
    for m in re.finditer(r'<\|channel\|>([A-Za-z_][A-Za-z0-9_]*)\s*<\|message\|>', text):
        s = m.end(); e = len(text)
        for t in ('<|return|>','<|end|>','<|call|>','<|start|>'):
            i = text.find(t, s)
            if i != -1: e = min(e, i)
        if m.group(1) == 'final': last = text[s:e]
    return last

# Warm-up (excluded from the gate; first call pays CUDA/kernel JIT costs).
_ = tokenizer(render(PROMPTS[0]['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
with torch.inference_mode():
    model.generate(**_ , max_new_tokens=16, do_sample=False,
                   pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print('warm-up done')

rows = []
for i, item in enumerate(PROMPTS):
    t_start = time.time()
    ids = tokenizer(render(item['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
    prompt_tokens = int(ids['input_ids'].shape[1])
    torch.cuda.synchronize(); t_prefill = time.time()
    with torch.inference_mode():                       # FIX 3
        out = model.generate(**ids, max_new_tokens=3072, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                             stopping_criteria=StoppingCriteriaList([HarmonyStop(STOP_IDS)]))
    torch.cuda.synchronize(); t_gen = time.time()
    new_ids = out[0][prompt_tokens:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=False)
    t_ext = time.time()
    ans = final_channel(raw)
    t_end = time.time()
    gen_tokens = int(new_ids.shape[0])
    rows.append({'item_id': item['item_id'], 'prompt_tokens': prompt_tokens,
        'generated_tokens': gen_tokens,
        'prefill_seconds': round(t_prefill - t_start, 3),
        'generation_seconds': round(t_gen - t_prefill, 3),
        'tokens_per_second': round(gen_tokens / max(t_gen - t_prefill, 1e-6), 3),
        'extraction_seconds': round(t_end - t_ext, 4),
        'total_seconds': round(t_end - t_start, 3),
        'termination_reason': 'STOP_TOKEN' if int(new_ids[-1]) in STOP_IDS else 'MAX_NEW_TOKENS',
        'final_channel_found': ans is not None, 'ok': ans is not None})
    r = rows[-1]
    print('ITEM %d/%d id=%s prompt_tok=%d gen_tok=%d prefill=%.2fs gen=%.2fs tok/s=%.2f total=%.2fs term=%s final=%s ok=%s'
          % (i+1, len(PROMPTS), r['item_id'][:12], r['prompt_tokens'], r['generated_tokens'],
             r['prefill_seconds'], r['generation_seconds'], r['tokens_per_second'],
             r['total_seconds'], r['termination_reason'], r['final_channel_found'], r['ok']))
    (WORKING/'benchmark-rows.jsonl').write_text(
        '\n'.join(json.dumps(x) for x in rows)+'\n', encoding='utf-8')

totals = sorted(r['total_seconds'] for r in rows)
median = totals[len(totals)//2]
tps = sum(r['tokens_per_second'] for r in rows)/len(rows)
verdict = 'GREEN' if median <= 180 else ('YELLOW' if median <= 300 else 'RED')
summary = {'MODEL_LOAD_SECONDS': round(MODEL_LOAD,2), 'ADAPTER_LOAD_SECONDS': round(ADAPTER_LOAD,2),
  'GPU_LAYOUT': str(getattr(model,'hf_device_map','n/a')),
  'PEAK_GPU_MEMORY_GiB': round(torch.cuda.max_memory_allocated()/1024**3, 3),
  'rows': rows, 'median_item_seconds': round(median,3),
  'mean_tokens_per_second': round(tps,3), 'final_channel_pass': sum(1 for r in rows if r['final_channel_found']),
  'VERDICT': verdict}
(WORKING/'benchmark-summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k!='rows'}, indent=2))
print('COMPLETE')
